# EHG Preterm Birth Prediction — VMD Pipeline
**Based on:** Bandpass → VMD → Energy-Based IMF Reconstruction → Feature Extraction → RF & SVM with SMOTE-in-fold

**Dataset:** Term-Preterm EHG Database (TPEHGDB v1.0.1) — 300 records, 3-channel, 20 Hz

---
### Pipeline Overview
1. Load `.dat` / `.hea` files → parse clinical metadata
2. Bandpass filter (0.3–3.0 Hz) per channel
3. Accumulate 3 channels into one signal
4. Apply VMD → decompose into K IMFs
5. Energy-based IMF selection → reconstruct signal
6. Feature extraction (Time / Frequency / Hjorth)
7. EDA + plots
8. Drop leakage columns, encode, impute, scale
9. Train/Test split (stratified)
10. RF & SVM — SMOTE applied **inside** each CV fold
11. Evaluation: AUC, classification report, confusion matrix

## STEP 0 — Install & Import Libraries

In [ ]:
# Install required packages
!pip install wfdb vmdpy antropy imbalanced-learn -q

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import antropy as ant
import wfdb
warnings.filterwarnings('ignore')

from vmdpy import VMD
from scipy import signal as sp_signal
from scipy.fft import fft, fftfreq
from collections import Counter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print('✅ All libraries imported')

## STEP 1 — Mount Drive & Locate Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/term-preterm-ehg-database-1.0.1/tpehgdb/'
assert os.path.isdir(DATA_DIR), f'Folder not found: {DATA_DIR}'

dat_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.dat')))
hea_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.hea')))
print(f'📂 .dat files : {len(dat_files)}')
print(f'📂 .hea files : {len(hea_files)}')

## STEP 2 — Parse Metadata from .hea Files

In [ ]:
def parse_hea(hea_path):
    record_id = os.path.splitext(os.path.basename(hea_path))[0]
    fields = {'record_id': record_id}
    field_map = {
        'Gestation': 'gestation',
        'Rectime': 'rectime',
        'Age': 'age',
        'Parity': 'parity',
        'Abortions': 'abortions',
        'Weight': 'weight',
        'Hypertension': 'hypertension',
        'Diabetes': 'diabetes',
        'Placental_position': 'placental_position',
        'Bleeding_first_trimester': 'bleeding_1st',
        'Bleeding_second_trimester': 'bleeding_2nd',
        'Funneling': 'funneling',
        'Smoker': 'smoker',
    }
    with open(hea_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('#'):
                continue
            content = line.lstrip('#').strip()
            parts = content.split(None, 1)
            if len(parts) < 2:
                continue
            key, val = parts[0], parts[1].strip()
            if key in field_map:
                fields[field_map[key]] = val
    return fields

meta_rows = [parse_hea(f) for f in hea_files]
meta_df = pd.DataFrame(meta_rows)

for col in ['gestation', 'rectime', 'age', 'parity', 'abortions', 'weight']:
    meta_df[col] = pd.to_numeric(meta_df[col], errors='coerce')

# Label: 1 = preterm (gestation < 37 weeks), 0 = term
meta_df['label'] = (meta_df['gestation'] < 37).astype(int)

print(f'Metadata shape: {meta_df.shape}')
print(f'\nLabel distribution:')
print(meta_df['label'].value_counts())
meta_df.head(3)

## STEP 3 — VMD Signal Processing Functions

In [ ]:
# ─── VMD Parameters (from research paper) ───────────────────────────
VMD_K     = 5      # number of IMFs to decompose into
VMD_ALPHA = 2000   # bandwidth constraint (moderate: preserves EHG bands)
VMD_TAU   = 0.0    # noise tolerance (0 = strict)
VMD_DC    = 0      # no DC component forced
VMD_INIT  = 1      # initialize omegas uniformly
VMD_TOL   = 1e-7   # convergence tolerance
FS        = 20     # sampling frequency (Hz)

# ─── Step A: Bandpass filter per channel (0.3–3.0 Hz) ────────────────
def bandpass_filter(sig, fs=FS, low=0.3, high=3.0):
    sig = np.nan_to_num(sig.astype(np.float64))
    nyq = fs / 2
    b, a = sp_signal.butter(4, [low / nyq, high / nyq], btype='band')
    return sp_signal.filtfilt(b, a, sig)

# ─── Step B: Accumulate 3 channels into one ──────────────────────────
def accumulate_channels(sig_array):
    """
    Paper: 'combined the filtered three-channel EHG into one signal
    using an accumulation operation' → simple channel-wise sum
    """
    acc = np.zeros(sig_array.shape[0])
    for i in range(min(3, sig_array.shape[1])):
        ch = sig_array[:, i]
        ch = np.nan_to_num(ch.astype(np.float64))
        acc += bandpass_filter(ch)
    return acc

# ─── Step C: VMD decomposition ───────────────────────────────────────
def apply_vmd(sig):
    """
    Returns IMFs array of shape (K, N)
    vmdpy signature: VMD(signal, alpha, tau, K, DC, init, tol)
    """
    try:
        u, _, _ = VMD(sig, VMD_ALPHA, VMD_TAU, VMD_K, VMD_DC, VMD_INIT, VMD_TOL)
        return u  # shape: (K, N)
    except Exception as e:
        # Fallback: return signal split into K equal-energy segments as IMFs
        print(f'  VMD failed: {e} — using fallback')
        N = len(sig)
        return np.array([sig * (1.0 / VMD_K)] * VMD_K)

# ─── Step D: Energy-based IMF selection & reconstruction ─────────────
def reconstruct_from_imfs(imfs):
    """
    Paper: 'energy-based intelligent signal reconstruction approach —
    combine IMFs having energy level above the computed threshold'
    Threshold = mean energy of all IMFs
    """
    energies = np.array([np.sum(imf**2) for imf in imfs])
    threshold = np.mean(energies)
    selected = imfs[energies >= threshold]
    if len(selected) == 0:
        selected = imfs  # fallback: use all
    reconstructed = np.sum(selected, axis=0)
    return reconstructed, energies, threshold

# ─── Full preprocessing for one record ───────────────────────────────
def preprocess_record(sig_array):
    """
    Returns: reconstructed signal, IMFs, energies, threshold
    """
    acc = accumulate_channels(sig_array)
    imfs = apply_vmd(acc)
    reconstructed, energies, threshold = reconstruct_from_imfs(imfs)
    return reconstructed, imfs, energies, threshold

print('✅ VMD processing functions defined')

## STEP 4 — Visualize VMD on One Sample Record

In [ ]:
# Load first record for visualization
sample_id = meta_df.iloc[0]['record_id']
sample_path = os.path.join(DATA_DIR, sample_id)
rec = wfdb.rdrecord(sample_path)
sig_array = rec.p_signal

# Preprocess
reconstructed, imfs, energies, threshold = preprocess_record(sig_array)
acc_signal = accumulate_channels(sig_array)

# ── Plot 1: Raw Accumulated vs Reconstructed ──────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
t = np.arange(len(acc_signal)) / FS

axes[0].plot(t, acc_signal, color='steelblue', linewidth=0.6)
axes[0].set_title('Accumulated EHG Signal (3-channel sum, bandpass 0.3–3.0 Hz)', fontsize=12)
axes[0].set_ylabel('Amplitude (mV)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, reconstructed, color='darkorange', linewidth=0.6)
axes[1].set_title(f'VMD-Reconstructed Signal (K={VMD_K}, energy-threshold IMF selection)', fontsize=12)
axes[1].set_ylabel('Amplitude (mV)')
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Record: {sample_id}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_01_accumulated_vs_reconstructed.png', dpi=120)
plt.show()

# ── Plot 2: All IMFs ──────────────────────────────────────────────────
fig, axes = plt.subplots(VMD_K, 1, figsize=(14, 2.5 * VMD_K), sharex=True)
for idx, (imf, eng) in enumerate(zip(imfs, energies)):
    color = 'crimson' if eng >= threshold else 'gray'
    label = f'IMF {idx+1}  |  Energy={eng:.3f}'
    if eng >= threshold:
        label += '  ✅ Selected'
    axes[idx].plot(t, imf, color=color, linewidth=0.5)
    axes[idx].set_ylabel(f'IMF {idx+1}', fontsize=9)
    axes[idx].set_title(label, fontsize=9)
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.suptitle(f'VMD IMFs — K={VMD_K} | Threshold={threshold:.4f}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_02_vmd_imfs.png', dpi=120)
plt.show()

# ── Plot 3: IMF Energy Bar Chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['crimson' if e >= threshold else 'lightgray' for e in energies]
bars = ax.bar([f'IMF {i+1}' for i in range(VMD_K)], energies, color=colors, edgecolor='black')
ax.axhline(threshold, color='navy', linestyle='--', linewidth=1.5, label=f'Threshold = {threshold:.4f}')
ax.set_title('IMF Energy Distribution (Red = Selected)', fontsize=12)
ax.set_ylabel('Energy')
ax.legend()
plt.tight_layout()
plt.savefig('plot_03_imf_energies.png', dpi=120)
plt.show()

print(f'\n📊 IMF Energies: {energies}')
print(f'📍 Threshold   : {threshold:.6f}')
print(f'✅ Selected IMFs: {np.sum(energies >= threshold)} / {VMD_K}')

## STEP 5 — Feature Extraction Functions

In [ ]:
# ─── Time-Domain Features ────────────────────────────────────────────
def time_features(sig, prefix='sig'):
    return {
        f'{prefix}_rms'        : float(np.sqrt(np.mean(sig**2))),
        f'{prefix}_std'        : float(np.std(sig)),
        f'{prefix}_mean'       : float(np.mean(sig)),
        f'{prefix}_mean_abs'   : float(np.mean(np.abs(sig))),
        f'{prefix}_variance'   : float(np.var(sig)),
        f'{prefix}_zcr'        : float(np.sum(np.diff(np.sign(sig)) != 0) / len(sig)),
        f'{prefix}_waveform_len': float(np.sum(np.abs(np.diff(sig)))),
        f'{prefix}_ptp'        : float(np.ptp(sig)),
        f'{prefix}_kurtosis'   : float(pd.Series(sig).kurtosis()),
        f'{prefix}_skewness'   : float(pd.Series(sig).skew()),
    }

# ─── Frequency-Domain Features ───────────────────────────────────────
def freq_features(sig, fs=FS, prefix='sig'):
    N = len(sig)
    freqs = fftfreq(N, d=1/fs)[:N//2]
    psd   = (2.0 / N * np.abs(fft(sig)[:N//2]))**2
    total = np.sum(psd) + 1e-10
    cum   = np.cumsum(psd)
    low_p = np.sum(psd[freqs < 1.0])
    hig_p = np.sum(psd[freqs >= 1.0])
    return {
        f'{prefix}_median_freq'   : float(freqs[np.searchsorted(cum, total * 0.5)]),
        f'{prefix}_mean_freq'     : float(np.sum(freqs * psd) / total),
        f'{prefix}_peak_freq'     : float(freqs[np.argmax(psd)]),
        f'{prefix}_spectral_energy': float(total),
        f'{prefix}_hl_freq_ratio' : float(hig_p / (low_p + 1e-10)),
    }

# ─── Hjorth Features ─────────────────────────────────────────────────
def hjorth_features(sig, prefix='sig'):
    act = np.var(sig)
    d1  = np.diff(sig)
    d2  = np.diff(d1)
    mob = np.sqrt(np.var(d1) / (act + 1e-10))
    cmp = np.sqrt(np.var(d2) / (np.var(d1) + 1e-10)) / (mob + 1e-10)
    return {
        f'{prefix}_hjorth_activity'  : float(act),
        f'{prefix}_hjorth_mobility'  : float(mob),
        f'{prefix}_hjorth_complexity': float(cmp),
    }

# ─── Extract features from one record ────────────────────────────────
def extract_features(sig_array, fs=FS):
    """
    Full pipeline: bandpass → accumulate → VMD → reconstruct → features
    Returns: dict of features
    """
    row = {}

    # 1. Preprocess
    reconstructed, imfs, energies, threshold = preprocess_record(sig_array)

    # 2. Features on reconstructed signal
    row.update(time_features(reconstructed, 'rec'))
    row.update(freq_features(reconstructed, fs, 'rec'))
    row.update(hjorth_features(reconstructed, 'rec'))

    # 3. Features on each selected IMF separately
    selected_imfs = imfs[energies >= threshold]
    for k, imf in enumerate(selected_imfs):
        p = f'imf{k+1}'
        row.update(time_features(imf, p))
        row.update(freq_features(imf, fs, p))
        row.update(hjorth_features(imf, p))

    # 4. IMF-level meta features
    row['n_selected_imfs']    = int(np.sum(energies >= threshold))
    row['total_imf_energy']   = float(np.sum(energies))
    row['energy_ratio_selected'] = float(
        np.sum(energies[energies >= threshold]) / (np.sum(energies) + 1e-10)
    )

    return row

print('✅ Feature extraction functions defined')

## STEP 6 — Extract Features for All Records (with Checkpoint)

In [ ]:
from joblib import Parallel, delayed
import time

CHECKPOINT_CSV = os.path.join(DATA_DIR, 'tpehg_vmd_features.csv')
N_JOBS         = 2

CLINICAL_COLS = ['age', 'parity', 'abortions', 'weight',
                 'hypertension', 'diabetes', 'placental_position',
                 'bleeding_1st', 'bleeding_2nd', 'funneling', 'smoker']

def process_one(row_tuple):
    _, row = row_tuple
    rec_id   = row['record_id']
    rec_path = os.path.join(DATA_DIR, rec_id)
    try:
        rec   = wfdb.rdrecord(rec_path)
        feats = extract_features(rec.p_signal)
        feats['record_id'] = rec_id
        feats['label']     = row['label']
        for col in CLINICAL_COLS:
            feats[col] = row.get(col, np.nan)
        return feats, rec_id, None
    except Exception as e:
        return None, rec_id, str(e)

# Resume from checkpoint if exists
if os.path.exists(CHECKPOINT_CSV):
    done_df  = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done_df['record_id'].tolist())
    all_rows = done_df.to_dict('records')
    print(f'♻️  Resuming: {len(done_ids)} records already done')
else:
    done_ids = set()
    all_rows = []
    print('🆕 Starting fresh')

pending = [(i, row) for i, row in meta_df.iterrows()
           if row['record_id'] not in done_ids]
print(f'📋 Pending: {len(pending)} | Done: {len(done_ids)}')

BATCH = 30
t0 = time.time()

for b_start in range(0, len(pending), BATCH):
    batch = pending[b_start : b_start + BATCH]
    results = Parallel(n_jobs=N_JOBS, prefer='threads')(
        delayed(process_one)(item) for item in batch
    )
    for feats, rec_id, err in results:
        if feats is not None:
            all_rows.append(feats)
        else:
            print(f'  ⚠️ Skipped {rec_id}: {err}')

    # Save checkpoint
    pd.DataFrame(all_rows).to_csv(CHECKPOINT_CSV, index=False)
    done_so_far = b_start + len(batch)
    elapsed = time.time() - t0
    print(f'  ✅ Processed {done_so_far}/{len(pending)} | {elapsed:.0f}s elapsed')

features_df = pd.DataFrame(all_rows)
print(f'\n🎯 Feature matrix shape: {features_df.shape}')
print(f'Label distribution:\n{features_df["label"].value_counts()}')

## STEP 7 — EDA & Plots

In [ ]:
df = features_df.copy()

# ── Plot 4: Class Distribution ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
df['label'].value_counts().plot(kind='bar', ax=ax,
    color=['steelblue', 'crimson'], edgecolor='black')
ax.set_xticklabels(['Term (0)', 'Preterm (1)'], rotation=0)
ax.set_title('Class Distribution', fontsize=12)
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width()/2, p.get_height() + 1),
                ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('plot_04_class_distribution.png', dpi=120)
plt.show()

# ── Plot 5: Missing Value Heatmap ─────────────────────────────────────
missing_pct = df.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(missing_pct) > 0:
    fig, ax = plt.subplots(figsize=(max(10, len(missing_pct)//3), 4))
    missing_pct.plot(kind='bar', ax=ax, color='tomato', edgecolor='black')
    ax.set_title('Missing Value % per Column', fontsize=12)
    ax.set_ylabel('Missing %')
    ax.tick_params(axis='x', rotation=90)
    plt.tight_layout()
    plt.savefig('plot_05_missing_values.png', dpi=120)
    plt.show()
else:
    print('✅ No missing values in extracted features')

# ── Plot 6: Feature Distributions — key features by class ─────────────
key_feats = ['rec_rms', 'rec_median_freq', 'rec_hjorth_activity',
             'rec_variance', 'rec_hl_freq_ratio', 'energy_ratio_selected']
key_feats = [f for f in key_feats if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()
for ax, feat in zip(axes, key_feats):
    for lbl, color, name in zip([0, 1], ['steelblue', 'crimson'], ['Term', 'Preterm']):
        vals = df.loc[df['label'] == lbl, feat].dropna()
        ax.hist(vals, bins=25, alpha=0.5, color=color, label=name, edgecolor='none')
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Feature Distributions by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_06_feature_distributions.png', dpi=120)
plt.show()

# ── Plot 7: Correlation Heatmap (numeric features only) ──────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_df  = df[num_cols].corr()
# Keep top 20 features most correlated with label
top20 = corr_df['label'].abs().nlargest(20).index.tolist()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_df.loc[top20, top20], ax=ax, cmap='coolwarm',
            center=0, annot=False, linewidths=0.4)
ax.set_title('Correlation Heatmap — Top 20 Features', fontsize=12)
plt.tight_layout()
plt.savefig('plot_07_correlation_heatmap.png', dpi=120)
plt.show()

print('✅ EDA plots saved')

## STEP 8 — Data Preprocessing (Leakage Removal, Encoding, Imputation, Scaling)

In [ ]:
df = features_df.copy()

# ─── 8.1 Drop leakage columns ─────────────────────────────────────────
# gestation  → directly encodes the label (preterm if gestation < 37)
# rectime    → highly correlated with gestation
# record_id  → identifier, not a feature
LEAKAGE_COLS = ['gestation', 'rectime', 'record_id']
existing_leakage = [c for c in LEAKAGE_COLS if c in df.columns]
df.drop(columns=existing_leakage, inplace=True)
print(f'✅ Dropped leakage columns: {existing_leakage}')

# ─── 8.2 Drop all-NaN columns ─────────────────────────────────────────
all_nan_cols = [c for c in df.columns if df[c].isna().all()]
df.drop(columns=all_nan_cols, inplace=True)
print(f'✅ Dropped all-NaN columns: {len(all_nan_cols)} — {all_nan_cols}')

# ─── 8.3 Encoding categorical features ───────────────────────────────
# Binary: yes/no → 1/0
binary_cols = ['hypertension', 'diabetes', 'bleeding_1st', 'bleeding_2nd', 'smoker']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col] = df[col].map({'yes': 1, 'no': 0, '1': 1, '0': 0})
        print(f'  Binary encoded: {col}')

# Ordinal: funneling → negative=0, none=1, positive=2
if 'funneling' in df.columns:
    df['funneling'] = df['funneling'].astype(str).str.strip().str.lower()
    df['funneling'] = df['funneling'].map({'negative': 0, 'none': 1, 'positive': 2})
    print('  Ordinal encoded: funneling')

# One-hot: placental_position (nominal, multiple categories)
if 'placental_position' in df.columns:
    df['placental_position'] = df['placental_position'].astype(str).str.strip().str.lower()
    pp_dummies = pd.get_dummies(df['placental_position'], prefix='pp', drop_first=False)
    df = pd.concat([df.drop(columns=['placental_position']), pp_dummies], axis=1)
    print(f'  One-hot encoded: placental_position → {list(pp_dummies.columns)}')

# ─── 8.4 Separate X and y ────────────────────────────────────────────
y = df['label'].values
X = df.drop(columns=['label'])

# ─── 8.5 Impute missing values ───────────────────────────────────────
# Identify numeric vs boolean/dummy columns
bool_cols = [c for c in X.columns if X[c].dropna().isin([0, 1]).all()]
num_cols  = [c for c in X.columns if c not in bool_cols]

num_imputer  = SimpleImputer(strategy='median')
bool_imputer = SimpleImputer(strategy='most_frequent')

X_num  = pd.DataFrame(num_imputer.fit_transform(X[num_cols]),   columns=num_cols,  index=X.index)
X_bool = pd.DataFrame(bool_imputer.fit_transform(X[bool_cols]), columns=bool_cols, index=X.index)
X = pd.concat([X_num, X_bool], axis=1)

print(f'\n✅ After preprocessing:')
print(f'   X shape : {X.shape}')
print(f'   y dist  : {Counter(y)}')
print(f'   NaN remaining: {X.isna().sum().sum()}')

## STEP 9 — Train / Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # maintain class ratio in both splits
)

print(f'Train: {X_train.shape} | Term={sum(y_train==0)}, Preterm={sum(y_train==1)}')
print(f'Test : {X_test.shape}  | Term={sum(y_test==0)}, Preterm={sum(y_test==1)}')

## STEP 10 — Model Training: RF & SVM with SMOTE inside each CV Fold

In [ ]:
from sklearn.metrics import make_scorer

# ─── Pipeline builder ────────────────────────────────────────────────
def build_pipeline(clf):
    """
    Order: StandardScaler → SMOTE → Classifier
    All inside imblearn Pipeline so SMOTE only sees train folds.
    """
    return ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote',  SMOTE(random_state=42, k_neighbors=3)),
        ('clf',    clf),
    ])

# ─── Classifiers ─────────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight={0: 1, 1: 7}   # ~7:1 imbalance
)

svm_clf = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    class_weight={0: 1, 1: 7},
    random_state=42
)

rf_pipeline  = build_pipeline(rf_clf)
svm_pipeline = build_pipeline(svm_clf)

# ─── Cross-validation (5-fold stratified) ────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cross_validate_pipeline(pipeline, X_tr, y_tr, name):
    print(f'\n🔄 Cross-validating: {name} ...')
    auc_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_fold_tr, X_fold_val = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        y_fold_tr, y_fold_val = y_tr[tr_idx], y_tr[val_idx]

        pipeline.fit(X_fold_tr, y_fold_tr)
        proba = pipeline.predict_proba(X_fold_val)[:, 1]
        auc   = roc_auc_score(y_fold_val, proba)
        auc_scores.append(auc)
        print(f'   Fold {fold+1}: AUC = {auc:.4f}')

    print(f'   📊 CV AUC = {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}')
    return auc_scores

rf_cv_aucs  = cross_validate_pipeline(rf_pipeline,  X_train, y_train, 'Random Forest')
svm_cv_aucs = cross_validate_pipeline(svm_pipeline, X_train, y_train, 'SVM')

# ─── Final training on full train set ────────────────────────────────
print('\n🏋️ Training final models on full train set...')
rf_pipeline.fit(X_train, y_train)
svm_pipeline.fit(X_train, y_train)
print('✅ Training done')

## STEP 11 — Evaluation on Test Set

In [ ]:
THRESHOLD = 0.35  # lower threshold to improve preterm recall

def evaluate_model(pipeline, X_te, y_te, name, cv_aucs):
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')

    proba      = pipeline.predict_proba(X_te)[:, 1]
    y_pred     = (proba >= THRESHOLD).astype(int)
    test_auc   = roc_auc_score(y_te, proba)

    print(f'  CV  AUC : {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}')
    print(f'  Test AUC: {test_auc:.4f}')
    print(f'\n  Classification Report (threshold={THRESHOLD}):')
    print(classification_report(y_te, y_pred, target_names=['Term', 'Preterm']))

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_te, proba)
    return fpr, tpr, test_auc, y_pred, proba

rf_fpr,  rf_tpr,  rf_auc,  rf_pred,  rf_proba  = evaluate_model(rf_pipeline,  X_test, y_test, 'Random Forest', rf_cv_aucs)
svm_fpr, svm_tpr, svm_auc, svm_pred, svm_proba = evaluate_model(svm_pipeline, X_test, y_test, 'SVM',           svm_cv_aucs)

# ── Plot 8: ROC Curves ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(rf_fpr,  rf_tpr,  color='darkorange', lw=2, label=f'Random Forest (AUC={rf_auc:.3f})')
ax.plot(svm_fpr, svm_tpr, color='steelblue',  lw=2, label=f'SVM (AUC={svm_auc:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — VMD Pipeline', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_08_roc_curves.png', dpi=120)
plt.show()

# ── Plot 9: Confusion Matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_pred, title in zip(
    axes,
    [rf_pred, svm_pred],
    ['Random Forest', 'SVM']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Term', 'Preterm'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title} (thr={THRESHOLD})', fontsize=11)

plt.suptitle('Confusion Matrices on Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_09_confusion_matrices.png', dpi=120)
plt.show()

# ── Plot 10: CV AUC Comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(1, 6)
ax.plot(x, rf_cv_aucs,  'o-', color='darkorange', label='Random Forest')
ax.plot(x, svm_cv_aucs, 's-', color='steelblue',  label='SVM')
ax.axhline(np.mean(rf_cv_aucs),  color='darkorange', linestyle='--', alpha=0.5)
ax.axhline(np.mean(svm_cv_aucs), color='steelblue',  linestyle='--', alpha=0.5)
ax.set_xlabel('Fold')
ax.set_ylabel('ROC-AUC')
ax.set_title('Cross-Validation AUC per Fold', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_10_cv_auc_per_fold.png', dpi=120)
plt.show()

print('\n✅ Evaluation complete. All plots saved.')

## STEP 12 — Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'Model'         : 'Random Forest',
        'CV_AUC_Mean'   : round(np.mean(rf_cv_aucs), 4),
        'CV_AUC_Std'    : round(np.std(rf_cv_aucs),  4),
        'Test_AUC'      : round(rf_auc, 4),
        'Denoising'     : 'VMD (K=5, energy-threshold)',
        'SMOTE'         : 'In-fold SMOTE',
        'Threshold'     : THRESHOLD,
    },
    {
        'Model'         : 'SVM (RBF)',
        'CV_AUC_Mean'   : round(np.mean(svm_cv_aucs), 4),
        'CV_AUC_Std'    : round(np.std(svm_cv_aucs),  4),
        'Test_AUC'      : round(svm_auc, 4),
        'Denoising'     : 'VMD (K=5, energy-threshold)',
        'SMOTE'         : 'In-fold SMOTE',
        'Threshold'     : THRESHOLD,
    },
])

print('\n📊 Final Results Summary')
print('='*70)
print(summary.to_string(index=False))
print('\n📌 Baseline (Fele-Žorž 2008, same dataset): AUC ≈ 0.60')